# CottonLens AI · Colab
Select **T4 GPU**, then **Run all**. This single notebook mounts Drive, creates isolated Python 3.12, runs mandatory compatibility tests, validates sources, trains and verifies the exact release. System Python packages are untouched. Smoke failure stops training; full errors remain visible. Use one active Colab session per Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys
REPO = Path('/content/CottonLensAI')
URL = 'https://github.com/ErayKulkizaga/CottonLensAI.git'
def git(*args):
    result = subprocess.run(['git', *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    if result.returncode:
        raise RuntimeError('Git failed; see full output above.')
    return result.stdout.strip()
if (REPO / '.git').exists():
    assert git('-C', REPO, 'remote', 'get-url', 'origin') == URL, 'Unexpected repository remote'
    assert not git('-C', REPO, 'status', '--porcelain'), 'Save your Colab repo edits before updating'
    git('-C', REPO, 'pull', '--ff-only')
else:
    git('clone', URL, REPO)
sys.path.insert(0, str(REPO / 'ml'))
import importlib, colab_setup
importlib.reload(colab_setup)
from colab_setup import setup, run, INFERENCE
git('-C', REPO, 'rev-parse', 'HEAD')

In [ ]:
PYTHON = setup()  # uv sync --locked; Python 3.12.11; dedicated /content venvs
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')

In [ ]:
SMOKE_PASSED = False
SMOKE_REPORT = DRIVE_ROOT / 'reports/environment-smoke.json'
run([PYTHON, '-m', 'cottonlens_ml.smoke', '--repo', REPO,
     '--report', SMOKE_REPORT, '--inference-python', INFERENCE / 'bin/python'])
SMOKE_PASSED = True

In [ ]:
DATA_PASSED = False
assert SMOKE_PASSED, 'Run the smoke cell successfully first'
run([PYTHON, '-m', 'cottonlens_ml.prepare', '--drive-root', DRIVE_ROOT, '--refresh'])
DATA_PASSED = True

In [ ]:
import json
TRAIN_PASSED = False
assert SMOKE_PASSED and DATA_PASSED, 'Run smoke and source validation first'
assert json.loads(SMOKE_REPORT.read_text())['status'] == 'passed', 'Smoke tests must pass first'
# Uses the sources just validated; does not download a second dataset.
run([PYTHON, '-m', 'cottonlens_ml.pipeline', '--drive-root', DRIVE_ROOT])
TRAIN_PASSED = True

In [ ]:
assert TRAIN_PASSED, 'Training/export must finish successfully first'
run([PYTHON, '-m', 'cottonlens_ml.validate_release', '--repo', REPO,
     '--drive-root', DRIVE_ROOT, '--inference-python', INFERENCE / 'bin/python'])
release = (DRIVE_ROOT / 'artifacts/releases/latest.txt').read_text().strip()
print('Validated ZIP:', DRIVE_ROOT / 'artifacts/releases' / release)
print('Checksum:', DRIVE_ROOT / 'artifacts/releases' / (release + '.sha256'))